In [ ]:
print("lets go")

lets go


Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters \
             langchain-huggingface sentence-transformers faiss-cpu \
             transformers pypdf accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


Upload Your Sanskrit Documents

In [ ]:
from google.colab import files

print("Upload your .txt or .pdf Sanskrit documents")
uploaded = files.upload()

doc_files = list(uploaded.keys())
print(f"Uploaded: {doc_files}")

Upload your .txt or .pdf Sanskrit documents


Saving Sanskrit_RAG.pdf to Sanskrit_RAG.pdf
Uploaded: ['Sanskrit_RAG.pdf']


 Load & Preprocess Documents

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []

for fname in doc_files:
    if fname.endswith(".pdf"):
        loader = PyPDFLoader(fname)
    else:
        loader = TextLoader(fname, encoding="utf-8")
    docs.extend(loader.load())

print(f"Loaded {len(docs)} document(s)")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["।", "\n\n", "\n", " "]   # Sanskrit-aware separators
)
chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")

Loaded 10 document(s)
Total chunks: 25


Create Embeddings & FAISS Index

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model (CPU)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"}
)

print("Building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("sanskrit_faiss_index")
print("✅ Index saved!")

Loading embedding model (CPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index...
✅ Index saved!


 Load CPU-based LLM

In [ ]:
from transformers import pipeline

print("Loading LLM (this may take a minute)...")
llm_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",   # Swap for flan-t5-large for better quality
    device=-1                       # -1 = CPU
)
print("✅ LLM ready!")

Loading LLM (this may take a minute)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ LLM ready!


Build the RAG Query Function

In [ ]:
def retrieve_and_answer(query: str, top_k: int = 3):
    print(f"\n🔍 Query: {query}\n")

    # Step 1: Retrieve relevant chunks
    retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
    relevant_docs = retriever.invoke(query)

    # Step 2: Build context
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    print("📄 Retrieved Context:\n", context[:600], "...\n")

    # Step 3: Generate answer
    prompt = f"""Answer the question based on the Sanskrit text context below.

Context:
{context}

Question: {query}
Answer:"""

    result = llm_pipeline(prompt, max_new_tokens=200, truncation=True)
    answer = result[0]["generated_text"]

    print("💬 Answer:\n", answer)
    return answer

Run Queries

In [ ]:
# English query
retrieve_and_answer("Who is Shankhanaada and what mistake did he make?")


🔍 Query: Who is Shankhanaada and what mistake did he make?

📄 Retrieved Context:
 । गृहे गिमˈामः, इित । The scholar did not know that the 
verb "badh" was 'Atmanep adi' (and hence had used the wrong 
form bAdhati instead of the correct form bAdhate') He thought, 
if even the palakhicarriers in this kingdom know the language so 
well, then meeting with the scholars shall certainly lead to my 
defeat. then he ordered kAlIdAsa, i really do not want to go to 
this extremely cold place. we should go back home."

चतुर˟  कालीदास˟  । Of the clever kAlIdAsa by: Kedar Naphade,                                       
ksn2@lehigh.edu  
 
 
घोिषतं कदािचत् भोजराǒा, यिद कोऽिप किवः मम दरबार ...

💬 Answer:
 The scholar did not know that the verb "badh" was 'Atmanep adi' (and hence had used the wrong form bAdhati instead of the correct form bAdhate') He thought, if even the palakhicarriers in this kingdom know the language so well, then meeting with the scholars shall certainly lead to my defeat. then h

'The scholar did not know that the verb "badh" was \'Atmanep adi\' (and hence had used the wrong form bAdhati instead of the correct form bAdhate\') He thought, if even the palakhicarriers in this kingdom know the language so well, then meeting with the scholars shall certainly lead to my defeat. then he ordered kAlIdAsa, i really do not want to go to this extremely cold place. we should go back home."'

In [ ]:
# Story moral
retrieve_and_answer("What is the moral of the story about the devotee?")


🔍 Query: What is the moral of the story about the devotee?

📄 Retrieved Context:
 । 
अथैकदा कʮन चोरः घǵ ामेकां चोरियȕा वनं गतः, ʩ ाťण च हतः । 
तदा सा घǵ ा वने एव अपतत् । 
अɊİ˝न् िदने क े चन वानराः तũ आगछन् । क ु तुहलेन तां घǵ ां हˑ े

सह । तİ˝न् काले िशिशरः भवित ऋतुः, शीतः च पवनः देहं ताडयित इव 
। वदित पİǷतः, शीतं बŠ बाधित इित

। न खलु जानाित पİǷतः यत् 
कालीदासः एव सः । पालखी ं ˋȽ योः वहन् िनगŊतः कालीदासः पİǷतेन ...

💬 Answer:
                                                                                                    


'                                                                                                   '

In [ ]:
# Transliterated Sanskrit query
retrieve_and_answer("What did kAlidAsa do when the foreign scholar arrived?")


🔍 Query: What did kAlidAsa do when the foreign scholar arrived?

📄 Retrieved Context:
 । चतुरः कालीदासः ȕरया एव 
पुवदित, न तथा बाधते शीतं यथा बाधित बाधते " On the day the 
scholar arrived, kAlIdAsa disguised himself as a palanquin carrier 
and went to recieve him. The scholar did not know that he was 
indeed kAlIdAsa. Carrying the pAlakhl on his shoulders, kAlIdAsa 
set off with the scholar. It was winter that time and the cold 
wind was hitting the body. The scholar said, "The cold hurts very 
much," Clever kAlIdAs immediately retorted, "Cold does not

। तथा भोजराजा दरबार े अकथयत् एषः पİǷतः आगिमˈित 
इित । Everybody knows that poet kAlIdAsa was in King Bhoj's 
court. Once a fore ...

💬 Answer:
 disguised himself as a palanquin carrier


'disguised himself as a palanquin carrier'

In [ ]:
print("🪷 Sanskrit RAG System Ready! Type 'quit' to exit.\n")

while True:
    query = input("Enter your query (Sanskrit / English / Transliterated): ")
    if query.lower() == "quit":
        break
    retrieve_and_answer(query)

🪷 Sanskrit RAG System Ready! Type 'quit' to exit.

Enter your query (Sanskrit / English / Transliterated): एकः परमः देवभक्तः अस्ति । सः प्रतिदिने भक्त्या देवस्य प्रार्थनाम् करोति ।

🔍 Query: एकः परमः देवभक्तः अस्ति । सः प्रतिदिने भक्त्या देवस्य प्रार्थनाम् करोति ।

📄 Retrieved Context:
 । न खलु जानाित पİǷतः यत् 
कालीदासः एव सः । पालखी ं ˋȽ योः वहन् िनगŊतः कालीदासः पİǷतेन

। 
अथैकदा कʮन चोरः घǵ ामेकां चोरियȕा वनं गतः, ʩ ाťण च हतः । 
तदा सा घǵ ा वने एव अपतत् । 
अɊİ˝न् िदने क े चन वानराः तũ आगछन् । क ु तुहलेन तां घǵ ां हˑ े

सह । तİ˝न् काले िशिशरः भवित ऋतुः, शीतः च पवनः देहं ताडयित इव 
। वदित पİǷतः, शीतं बŠ बाधित इित ...

💬 Answer:
                                                                                                    
Enter your query (Sanskrit / English / Transliterated): what message sent to King Bhoj

🔍 Query: what message sent to King Bhoj

📄 Retrieved Context:
 । तथा भोजराजा दरबार े अकथयत् एषः पİǷतः आगिमˈित 
इित । Everybody knows that poet kAlIdAsa was in King Bhoj's 
co